In [ ]:
import json
import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
DATASET_PATH = "synthetic_outputs.jsonl"

### Loading the dataset

In [ ]:
data = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f.readlines():
        data.append(json.loads(line))

df = pd.DataFrame(data)
df.head()

### Visualize the dataset

#### Model counts

In [ ]:
# Count each model's occurrences in the dataset

counts_a = df['model_a'].apply(lambda x: x.get('model_name')).value_counts()
counts_b = df['model_b'].apply(lambda x: x.get('model_name')).value_counts()

model_counts = counts_a.add(counts_b, fill_value=0).astype(int).sort_values(ascending=False)

print(f"Conversation items: {model_counts.sum() // 2}")

# Plot the resulting charts

colours = plt.cm.Paired(range(len(model_counts)))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 8))

model_counts.plot.bar(ax=ax1, color=colours, edgecolor='black', width=0.7)

ax1.set_xlabel('', fontsize=12)
ax1.set_ylabel('Number of conversation items', fontsize=12)
ax1.set_xticklabels(model_counts.index, rotation=45, ha='right', fontsize=10)

for i, v in enumerate(model_counts):
    ax1.text(i, v + max(model_counts) * 0.01, str(v), ha='center', va='bottom', fontsize=10, fontweight='bold')

model_counts.plot.pie(ax=ax2, autopct='%1.1f%%', startangle=90, colors=colours, wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}, textprops={'fontsize': 12})

ax2.set_ylabel('')

fig.suptitle("Distribution of conversation items per model", fontsize=14, fontweight='bold', y=0.98)

plt.tight_layout()
plt.show()

#### Completion token count

In [ ]:
tokens = {model: 0 for model in df["model_a"].apply(lambda x: x.get("model_name")).unique()}

for idx, row in df.iterrows():
    model_a = row["model_a"].get("model_name")
    model_b = row["model_b"].get("model_name")
    
    usage_a = row["model_a"]["llm_usage"]
    usage_b = row["model_b"]["llm_usage"]
    
    if usage_a is not None:
        tokens[model_a] += usage_a.get("completion_tokens", 0)
    
    if usage_b is not None:
        tokens[model_b] += usage_b.get("completion_tokens", 0)

In [ ]:
pd.options.display.float_format = '{: .0f}'.format
out = pd.DataFrame(list(tokens.items()), columns=["Model", "Completion Tokens"]).sort_values(by="Completion Tokens", ascending=False)

avg_tokens_per_completion = out["Completion Tokens"] / model_counts[out["Model"]].values
out["Average Tokens per Completion"] = avg_tokens_per_completion

total_tokens = out["Completion Tokens"].sum()
out["Percentage"] = (out["Completion Tokens"] / total_tokens) * 100

min_tokens = out["Completion Tokens"].min()
out["Factor"] = out["Completion Tokens"] / min_tokens

out['Completion Tokens'] = out['Completion Tokens'].apply('{:,}'.format)
out['Average Tokens per Completion'] = out['Average Tokens per Completion'].apply('{:,.2f}'.format)
out['Percentage'] = out['Percentage'].apply(lambda x: f"{x:.1f} %")
out['Factor'] = out['Factor'].apply(lambda x: f"{x:.2f}")

display(out)

Some of the models, mostly frontier ones (`deepseek/deepseek-v4-pro`, `mistralai/mistral-large-2512`, `deepseek/deepseek-v4-flash`, `openai/gpt-5.4`) consume much more output tokens than others (2- to 3-fold).

#### Analysis of prompt configurations

In [ ]:
# Count output tokens per prompt configuration

out = df.copy()

COMPLETION_TOKENS = "completion_tokens"

out["output_a"] = out["model_a"].str["llm_usage"].str[COMPLETION_TOKENS].fillna(0)
out["output_b"] = out["model_b"].str["llm_usage"].str[COMPLETION_TOKENS].fillna(0)

out["total_output_tokens"] = out["output_a"] + out["output_b"]

avg_per_config = out.groupby("prompt_configuration")["total_output_tokens"].mean()

print(avg_per_config)

#### Number of completions finished abnormally

In [ ]:
out = df.copy()

counts = out["model_a"].str["llm_usage"].str["finish_reason"].fillna("stop")
print(counts.value_counts())

# Learner corpus data

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import json
import numpy as np

with open("../data/reactions.json", 'r', encoding='utf-8') as f:
    df = json.load(f)

ids = [x["user_id"] for x in df]

# Number of items for each user
items_per_user = Counter(ids)

counts = list(items_per_user.values())

mean, sd = np.mean(counts), np.std(counts)
print(f"Mean: {mean:.2f}, Standard Deviation: {sd:.2f}")
print(f"Min: {np.min(counts)}, Max: {np.max(counts)}")

# Distribution: number of users having N items
distribution = Counter(items_per_user.values())
distribution.update({5: 0}) # Add missing value for 5 items to ensure it appears in the plot

x = sorted(distribution)
y = [distribution[n] for n in x]

plt.bar(x, y)

plt.xlabel("Number of conversations per participant (user ID)")
plt.ylabel("Number of participants")
plt.xticks(x)
plt.tight_layout()
plt.savefig("plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
with open("../data/reactions.json", 'r', encoding='utf-8') as f:
    df = json.load(f)

conv_turns = [x["conv_turns"] for x in df]

counts = list(conv_turns)

print(counts)

mean, sd = np.mean(counts), np.std(counts)
print(f"Mean: {mean:.2f}, Standard Deviation: {sd:.2f}")
print(f"Min: {np.min(counts)}, Max: {np.max(counts)}")

x = sorted(distribution)
y = [distribution[n] for n in x]

plt.bar(x, y)

plt.xlabel("Number of turns per conversation")
plt.ylabel("Number of conversations")
plt.xticks(x)
plt.tight_layout()
plt.savefig("plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
with open("../data/reactions.json", 'r', encoding='utf-8') as f:
    df = json.load(f)
    
preferences = [x["preferred_model"] for x in df]
ratings = [x["rating"] for x in df if x["rating"] != 0]
labels = [any(x["labels"].values()) for x in df if x["labels"] is not None and any(x["labels"].values())]
comments = [x["comment"] for x in df if x["comment"] is not None]

print(len(preferences), len(ratings), len(labels), len(comments))

print(f"Proportion of preferences: {len(preferences) / len(df) * 100:.2f}%")
print(f"Proportion of ratings: {len(ratings) / len(df) * 100:.2f}%")
print(f"Proportion of labels: {len(labels) / len(df) * 100:.2f}%")
print(f"Proportion of comments: {len(comments) / len(df) * 100:.2f}%")

In [ ]:
with open("../data/students_teacher_gold.json", 'r', encoding='utf-8') as f:
    df_original = json.load(f)
    
df = [x["teacher_annotation_data"] for x in df_original]
df = list([item for sublist in df for item in sublist])  # Flatten the list of lists

LABELS = ["complete", "correct", "relevant", "concise", "scaffolding", "understandable"]

preferences = [x["preferred_model"] for x in df]
ratings = [x["rating"] for x in df if x["rating"] != 0]
labels = [any(x[y] for y in LABELS) for x in df if any(x[y] for y in LABELS)]
comments = [x["comment"] for x in df if x["comment"] is not None]

print(len(preferences), len(ratings), len(labels), len(comments))

print(f"Proportion of preferences: {len(preferences) / len(df) * 100:.2f}%")
print(f"Proportion of ratings: {len(ratings) / len(df) * 100:.2f}%")
print(f"Proportion of labels: {len(labels) / len(df) * 100:.2f}%")
print(f"Proportion of comments: {len(comments) / len(df) * 100:.2f}%")